# backward-func-lookup — worked example 2: Add a has_back_func / try_get helper to the lookup

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-func-lookup`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The core `BackwardFuncLookup` raises `KeyError` on a missing `(fwd, argnum)` key. Sometimes a caller wants to *probe* whether a back fn exists without an exception — e.g. to decide if an op participates in backprop. A `has_back_func` predicate and a `try_get` returning `None` give that, both built on the same flat dict.

## Worked solution

**Step 1 — same flat storage.** `add_back_func` and `get_back_func` are unchanged: one dict, tuple keys.

**Step 2 — the predicate.** `has_back_func(fwd, argnum)` is just `(fwd, argnum) in self.back_funcs`. Membership test on a dict is `O(1)` and never raises, so it is safe to call on unregistered ops.

**Step 3 — the soft getter.** `try_get(fwd, argnum)` returns `self.back_funcs.get((fwd, argnum))`, which yields the stored fn or `None`. This is the non-raising twin of `get_back_func`; the strict version is still the one used inside the reverse pass where a missing registration is a real bug.

**Step 4 — exercise.** We register only `(t.exp, 0)`. `has_back_func(t.exp, 0)` is `True`; `has_back_func(t.exp, 1)` is `False` because we never registered argnum 1. `try_get(t.log, 0)` returns `None` since `t.log` was never registered — no exception. This separation lets the dispatcher branch on availability instead of catching errors.

In [ ]:
class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}

    def add_back_func(self, forward_fn, argnum, back_fn):
        self.back_funcs[(forward_fn, argnum)] = back_fn

    def get_back_func(self, forward_fn, argnum):
        key = (forward_fn, argnum)
        if key not in self.back_funcs:
            raise KeyError(f'No back_fn for ({forward_fn!r}, argnum={argnum}).')
        return self.back_funcs[key]

    def has_back_func(self, forward_fn, argnum):
        return (forward_fn, argnum) in self.back_funcs

    def try_get(self, forward_fn, argnum):
        return self.back_funcs.get((forward_fn, argnum))


def exp_back0(grad_out, out, x):
    return grad_out * out

BFL = BackwardFuncLookup()
BFL.add_back_func(t.exp, 0, exp_back0)

print('has (exp, 0):', BFL.has_back_func(t.exp, 0))
print('has (exp, 1):', BFL.has_back_func(t.exp, 1))
print('try_get (log, 0) is None:', BFL.try_get(t.log, 0) is None)
print('try_get (exp, 0) runs:', BFL.try_get(t.exp, 0)(t.ones(2), t.exp(t.zeros(2)), t.zeros(2)).tolist())